# 💻 Laboratório 17: Moldando o Tempo (Séries Temporais)

**Disciplina:** Extração e Preparação de Dados (IBM8915)  
**Professor:** Luís Aramis  

**Objetivo:** Transformar um registro diário ruidoso em uma base agregada limpa e aplicar a "Máquina do Tempo" (Lags) para que um algoritmo de Machine Learning consiga usar o passado para prever o futuro, sem cometer *Data Leakage*.

In [1]:
# Importação das bibliotecas essenciais
import pandas as pd
import numpy as np

# Simulando a carga de um Dataset de Vendas Diárias (1 ano inteiro de histórico)
# Em um cenário real, você usaria pd.read_csv('vendas_2025.csv')
np.random.seed(42)
datas = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
vendas_diarias = np.random.normal(loc=5000, scale=1500, size=len(datas)).round(2)

dados_vendas = pd.DataFrame({
    'Data': datas.astype(str), # Simulando que a data veio do CSV como String
    'Receita_Diaria': vendas_diarias
})

print("--- Base de Dados Bruta (Primeiros 5 dias) ---")
display(dados_vendas.head())

--- Base de Dados Bruta (Primeiros 5 dias) ---


,Data,Receita_Diaria
0,2025-01-01,5745.07
1,2025-01-02,4792.60
2,2025-01-03,5971.53
3,2025-01-04,7284.54
4,2025-01-05,4648.77


## Passo 1: O Paradigma da Ordem (O Índice do Tempo)

Diferente de clientes aleatórios no Titanic, em Séries Temporais a **ordem importa**. A data não pode ser apenas uma coluna comum; ela precisa ser o **Índice** (a espinha dorsal) do nosso DataFrame para que possamos fazer a matemática do tempo funcionar.

**Sua Tarefa:**
1. Converta a coluna de *String* para o tipo oficial `datetime` do Pandas.
2. Defina essa coluna convertida como o índice do DataFrame.

In [2]:
# 1. Converta a coluna 'Data' para o tipo datetime usando pd.to_datetime()
dados_vendas['Data'] = pd.to_datetime(dados_vendas['Data'])

# 2. Transforme a coluna 'Data' no índice do DataFrame usando .set_index()
# Dica: salve o resultado em uma nova variável chamada 'df_temporal'
df_temporal = dados_vendas.set_index('Data')

print("--- Dados com a Data no Índice ---")
display(df_temporal.head(3))

--- Dados com a Data no Índice ---


,Receita_Diaria
Data,
2025-01-01,5745.07
2025-01-02,4792.60
2025-01-03,5971.53


## Passo 2: Esmagando o Ruído (Resampling)

A diretoria não quer analisar a flutuação diária, pois ela é cheia de ruídos (dias de chuva, feriados no meio da semana). O objetivo de negócios é prever o **faturamento mensal**. 

Precisamos pegar nossa base de 365 dias e esmagá-la em 12 meses usando a agregação temporal do Pandas.

**Sua Tarefa:**
1. Use o método `.resample()` passando o parâmetro `'M'` (Mensal).
2. Encadeie a função matemática para somar (`.sum()`) o faturamento de cada mês.

In [3]:
# 1 e 2. Faça o resample mensal e a soma da Receita_Diaria
# Dica: df_temporal.resample(...).sum()
df_mensal = df_temporal.resample('M').sum() # Use 'ME' se estiver em uma versão muito recente do Pandas

# Para fins de clareza, vamos renomear a coluna de 'Receita_Diaria' para 'Receita_Mensal'
df_mensal = df_mensal.rename(columns={'Receita_Diaria': 'Receita_Mensal'})

print("--- Fechamento Mensal Consolidado ---")
display(df_mensal)

--- Fechamento Mensal Consolidado ---


/var/folders/7_/mlzr2t1j7rn6n62c72wwnrmh0000gn/T/ipykernel_7065/131622638.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_mensal = df_temporal.resample('M').sum() # Use 'ME' se estiver em uma versão muito recente do Pandas


,Receita_Mensal
Data,
2025-01-31,145630.80
2025-02-28,133986.92
2025-03-31,157043.11
2025-04-30,149088.68
2025-05-31,151102.61
2025-06-30,163108.14
2025-07-31,159572.85
2025-08-31,153998.55
2025-09-30,145901.85


## Passo 3: A Máquina do Tempo (Lags)

Um modelo de Machine Learning vai ler a linha de Dezembro e tentar prever o futuro. Mas como ele saberá o que aconteceu no mês passado se ele só enxerga a linha atual? 

Nós precisamos "empurrar" os dados do passado para o presente criando as famosas variáveis de *Lag* (atraso).

**Sua Tarefa:**
1. Crie uma nova coluna chamada `Receita_1_Mes_Atras` deslocando a Receita Mensal em 1 posição para baixo.
2. Crie uma nova coluna chamada `Receita_2_Meses_Atras` deslocando a Receita Mensal em 2 posições para baixo.

In [4]:
# 1. Crie o lag de 1 mês (Dica: use .shift(1))
df_mensal['Receita_1_Mes_Atras'] = df_mensal['Receita_Mensal'].shift(1)

# 2. Crie o lag de 2 meses (Dica: use .shift(2) na coluna original 'Receita_Mensal')
df_mensal['Receita_2_Meses_Atras'] = df_mensal['Receita_Mensal'].shift(2)

print("--- DataFrame com os Preditores do Passado ---")
display(df_mensal)

--- DataFrame com os Preditores do Passado ---


,Receita_Mensal,Receita_1_Mes_Atras,Receita_2_Meses_Atras
Data,,,
2025-01-31,145630.80,NaN,NaN
2025-02-28,133986.92,145630.80,NaN
2025-03-31,157043.11,133986.92,145630.80
2025-04-30,149088.68,157043.11,133986.92
2025-05-31,151102.61,149088.68,157043.11
2025-06-30,163108.14,151102.61,149088.68
2025-07-31,159572.85,163108.14,151102.61
2025-08-31,153998.55,159572.85,163108.14
2025-09-30,145901.85,153998.55,159572.85


## Passo 4: O Preço da Viagem no Tempo

Observe o DataFrame que você acabou de gerar. O que aconteceu com Janeiro e Fevereiro? 

Eles ganharam valores nulos (`NaN`)! Como empurramos os dados para o futuro, o primeiro mês não tem um "mês anterior" para consultar. Esse é o preço que pagamos por criar Lags.

Como algoritmos matemáticos odeiam nulos (e nós não podemos "inventar" uma receita para o ano anterior que não existe na base), a abordagem mais segura em Séries Temporais é eliminar essas primeiras linhas corrompidas.

**Sua Tarefa Final:**
Utilize o que aprendemos nas Aulas 06 e 07 e aplique o método `.dropna()` para limpar sua base antes de entregá-la para a IA.

In [5]:
# 1. Remova as linhas com valores nulos causados pelo shift
df_pronto_para_modelo = df_mensal.dropna()

print("✅ Base Perfeitamente Tratada para Machine Learning!")
display(df_pronto_para_modelo)

✅ Base Perfeitamente Tratada para Machine Learning!


,Receita_Mensal,Receita_1_Mes_Atras,Receita_2_Meses_Atras
Data,,,
2025-03-31,157043.11,133986.92,145630.80
2025-04-30,149088.68,157043.11,133986.92
2025-05-31,151102.61,149088.68,157043.11
2025-06-30,163108.14,151102.61,149088.68
2025-07-31,159572.85,163108.14,151102.61
2025-08-31,153998.55,159572.85,163108.14
2025-09-30,145901.85,153998.55,159572.85
2025-10-31,158022.34,145901.85,153998.55
2025-11-30,160465.72,158022.34,145901.85


### Critério de Êxito para o Portfólio (GitHub):
Se a sua base `df_pronto_para_modelo` começou exatamente no mês de **Março** (já que Jan e Fev foram deletados pelo `.dropna()`) e não possui nenhum dado nulo, sua "Máquina do Tempo" foi construída com sucesso e você garantiu que não haja vazamento de dados do futuro para o passado! 

Lembre-se de salvar e fazer o *commit* deste notebook no seu repositório para fecharmos a avaliação.